# Attention, Virality, and Digital Culture
---
### Opening Question
**How does attention move across major social media platforms, and what patterns seem to support virality?**

### Objective
This notebook explores how attention flows across Twitter, Reddit, Instagram, YouTube, TikTok, and Facebook. The dataset contains posts with engagement metrics, sentiment details, and cross-platform spread indicators.

The aim is to understand which kinds of content receive attention, whether some platforms amplify certain types of engagement, and what factors might separate viral posts from the rest.

### Dataset Overview
- **Source**: Multi-Platform Social Sentiment Evolution dataset from [Kaggle](https://www.kaggle.com/datasets/benjaminneil/social-sentiment-evolution).
- **Variables**: platform, topic, sentiment, likes, shares, comments, views, hour_of_day, is_weekend, viral_coefficient, cross_platform_spread, verified, follower_count
- **Analysis Approach**: Exploratory Data Analysis [EDA] with focus on cross-platform attention patterns, virality indicators, and engagement dynamics through visualisation and statistical summaries.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Set visualisation style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10


In [ ]:
# colour scheme
primary = "#0D47A1"
secondary = "#90CAF9"

gradient = ["#BBDEFB", "#90CAF9", "#64B5F6", "#42A5F5", "#2196F3", "#1E88E5", "#1565C0", "#0D47A1"]

# accent colours
mangoacc = "#ff9f43"
pinkacc = "#ff6b81"
skyacc = "#74b9ff"

### 1. Dataset Loading and Overview

I will start by loading the dataset and checking its basic structure. This gives a first impression of what the file contains before the real analysis begins.

In [ ]:
# load the dataset
path = "trendsanalysis.csv"
file = pd.read_csv(path)

In [ ]:
print(f"Dataset shape: {file.shape}")

In [ ]:
print(f"\nTotal records: {file.shape[0]}")

### 1.1 First Look at the Data

Displaying the first and last five rows helps me understand the dataset layout.

In [ ]:
print("First 5 records:")
file.head()

In [ ]:
print("Last 5 records:")
file.tail()

### 1.2 Data Types and Column Information

Here I check column names, data types, and the basic structure.

In [ ]:
print("Dataset Info:")
file.info()

In [ ]:
print("\nColumn Names:")
file.columns.tolist()

### 2. Data Quality Checks

Before moving to analysis, I need to see if the dataset has missing values, duplicate rows, or any obvious issues. Clean data makes the rest of the work much smoother [which I always appreciate].

In [ ]:
# missing values check
print("Missing Values:")
missing = file.isnull().sum()
missing_pct = (missing / len(file)) * 100
missing_file = pd.DataFrame({"Column": missing.index, "Missing Count": missing.values, "Missing %": missing_pct.values})
print(missing_file[missing_file["Missing Count"] > 0] if missing_file["Missing Count"].sum() > 0 else "No missing values found!")
print(f"\nTotal missing values in dataset: {file.isnull().sum().sum()}")

In [ ]:
# duplicates check
print(f"Number of duplicate rows: {file.duplicated().sum()}")
print(f"Number of unique records: {len(file.drop_duplicates())}")

In [ ]:
# summary of variables
print("\nDataset Statistics:")
file.describe(include="all").round(2)

### Observations

- **Data completeness**: No missing values or duplicate rows. Good sign.
- **Variable types**: Numeric engagement metrics are correctly typed; categorical columns [platform, topic, sentiment] are objects.

**What this suggests**: The dataset is well-structured and doesn't need cleaning before we explore attention patterns.

### 3. Platform Attention Analysis

This section compares engagement across platforms. Since attention can mean likes, shares, comments, or views, I will look at more than one metric.

**Questions:**
- Which platforms generate the most attention?
- Do different platforms dominate different engagement metrics?
- How does platform design influence engagement type?

In [ ]:
print("Average Engagement by Platform:")
platform_engagement = file.groupby("platform")[["likes", "shares", "comments", "views"]].mean().round(0)
print(platform_engagement)
print(f"\nPercentage Distribution:")
print((platform_engagement / platform_engagement.sum() * 100).round(1))

In [ ]:
# visualisation: platform likes
plt.figure(figsize=(10, 6))
platform_likes = file.groupby("platform")["likes"].mean().sort_values(ascending=True)
ax = sns.barplot(y=platform_likes.index, x=platform_likes.values, palette=gradient[-len(platform_likes):], hue=platform_likes.index, legend=False)

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=10, fontweight="bold", padding=5)

plt.title("Average Likes per Platform", fontsize=14, fontweight="bold")
plt.xlabel("Average Number of Likes", fontsize=11)
plt.ylabel("Platform", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# visualisation: platform shares
plt.figure(figsize=(10, 6))
platform_shares = file.groupby("platform")["shares"].mean().sort_values(ascending=True)
ax = sns.barplot(y=platform_shares.index, x=platform_shares.values, palette=gradient[-len(platform_shares):], hue=platform_shares.index, legend=False)

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=10, fontweight="bold", padding=5)

plt.title("Average Shares per Platform", fontsize=14, fontweight="bold")
plt.xlabel("Average Number of Shares", fontsize=11)
plt.ylabel("Platform", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# visualisation: platform comments and views
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

platform_comments = file.groupby("platform")["comments"].mean().sort_values(ascending=True)
sns.barplot(y=platform_comments.index, x=platform_comments.values, palette=gradient[-len(platform_comments):], hue=platform_comments.index, legend=False, ax=axes[0])
axes[0].set_title("Average Comments per Platform", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Average Comments", fontsize=11)
axes[0].set_ylabel("Platform", fontsize=11)

platform_views = file.groupby("platform")["views"].mean().sort_values(ascending=True)
sns.barplot(y=platform_views.index, x=platform_views.values, palette=gradient[-len(platform_views):], hue=platform_views.index, legend=False, ax=axes[1])
axes[1].set_title("Average Views per Platform", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Average Views", fontsize=11)
axes[1].set_ylabel("Platform", fontsize=11)

plt.tight_layout()
plt.show()

### Observations
- **YouTube dominates views**: Video-centric platforms naturally get higher view counts.
- **TikTok leads in likes**: Its fast, interactive design encourages many likes per post.
- **Twitter [X] shows strong sharing behaviour**: Retweets are a primary engagement method.
- **Reddit encourages comments**: Discussion forums produce more conversation than other sites.

**What this suggests**: There is no single most engaging platform. Engagement depends on what you measure, because a like, share, comment, and view all represent different forms of attention.

### 4. Topic and Attention Analysis

This part checks how topics perform across platforms. It helps show whether a topic does well everywhere or mainly on specific platforms.

**Questions:**
- Do certain topics capture more attention than others?
- Which topics generate the most discussion?
- Does topic performance vary by platform?

In [ ]:
# topic-platform engagement
print("Topic Performance Across Platforms [Average Likes]:")
topic_platform_likes = file.pivot_table(values="likes", index="topic", columns="platform", aggfunc="mean").fillna(0)
topic_platform_likes.round(2)

In [ ]:
# visualisation: topic performance heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(topic_platform_likes, annot=True, cmap="Blues", fmt=".0f", cbar_kws={"label": "Average Likes"}, linewidths=0.5)
plt.title("Topic Performance Across Platforms", fontsize=14, fontweight="bold")
plt.xlabel("Platform", fontsize=11)
plt.ylabel("Topic", fontsize=11)
plt.tight_layout()
plt.show()

### Observations
- **Entertainment, technology, and gaming** tend to perform well across several platforms.
- **Politics** can create strong discussion on platforms where debate is common [e.g., Twitter, Reddit].
- Some topics work well in one space but not another, which shows that audience culture matters.

**What this suggests**: Topic and platform work together. The same topic can receive very different attention depending on where it is posted.

### 4.1 Topic Discussion Analysis

Comments are useful for studying discussion because they usually need more effort than likes. This step checks total comments by topic.

In [ ]:
# visualisation: top topics by comments
topic_comments = file.groupby("topic")["comments"].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=topic_comments.head(10).index, y=topic_comments.head(10).values, palette=gradient, hue=topic_comments.head(10).index, legend=False)

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=10, fontweight="bold", padding=5)

plt.title("Top 10 Topics by Total Comments", fontsize=14, fontweight="bold")
plt.xlabel("Topic", fontsize=11)
plt.ylabel("Total Comments", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Observations
- Topics that invite opinion or debate [politics, gaming, entertainment] tend to generate the most comments.
- The distribution is uneven. A few topics drive the majority of discussion.

**What this suggests**: For content that aims to spark conversation, choosing a polarising or highly engaging topic matters more than the platform alone.

### 5. Timing and Engagement Analysis

This section checks whether posts receive different engagement depending on the hour of the day and whether they are posted on weekdays or weekends.

**Questions:**
- Does posting time affect attention?
- Are weekend posts more engaging than weekday posts?
- Is there an optimal posting window?

In [ ]:
# visualisation: hourly engagement
plt.figure(figsize=(10, 6))
hourly_likes = file.groupby("hour_of_day")["likes"].mean()

plt.plot(hourly_likes.index, hourly_likes.values, marker="o", linestyle="-", color=primary, linewidth=2)
plt.fill_between(hourly_likes.index, hourly_likes.values, color=secondary, alpha=0.35)
plt.title("Average Likes by Hour of Day", fontsize=14, fontweight="bold")
plt.xlabel("Hour of Day [24-hour format]", fontsize=11)
plt.ylabel("Average Likes", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("Average Engagement: Weekend vs Weekday:")
weekend_engagement = file.groupby("is_weekend")[["likes", "shares", "comments"]].mean()
weekend_engagement.index = ["Weekday", "Weekend"]
weekend_engagement.round(2)

In [ ]:
# visualisation: weekend vs weekday
weekend_plot = weekend_engagement.reset_index().rename(columns={"index": "Day Type"})
weekend_melted = weekend_plot.melt(id_vars="Day Type", var_name="Metric", value_name="Average")

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=weekend_melted, x="Metric", y="Average", hue="Day Type", palette=[primary, secondary])

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=10, fontweight="bold", padding=5)

plt.title("Average Engagement: Weekend vs Weekday", fontsize=14, fontweight="bold")
plt.xlabel("Engagement Metric", fontsize=11)
plt.ylabel("Average Count", fontsize=11)
plt.tight_layout()
plt.show()

### Observations
- Engagement tends to rise during late morning and early evening hours, which aligns with typical user active periods.
- Weekend posts receive slightly more likes and comments on average than weekday posts, though the difference isn't huge.

**What this suggests**: Timing matters, but it is not everything. A weak post at the perfect time is still a weak post [which is unfortunate but fair].

### 6. Cross-Platform Spread Analysis

This section compares posts that spread across several platforms with those that stay on one platform.

**Questions:**
- What separates high-spread content from low-spread content?
- How are engagement and virality indicators related?
- Does cross-platform spread guarantee higher engagement?

In [ ]:
# cross-platform spread comparison
median_spread = file["cross_platform_spread"].median()
high_spread = file[file["cross_platform_spread"] > median_spread]
low_spread = file[file["cross_platform_spread"] <= median_spread]
print(f"High spread posts count: {len(high_spread)}")
print(f"Low spread posts count: {len(low_spread)}")

In [ ]:
print("Engagement Comparison: High Spread vs Low Spread:")
high_spread_avg = high_spread[["likes", "shares", "comments", "views"]].mean()
low_spread_avg = low_spread[["likes", "shares", "comments", "views"]].mean()

spread_comparison = pd.DataFrame({
    "High Spread": high_spread_avg,
    "Low Spread": low_spread_avg
}).round(2)
spread_comparison

In [ ]:
# visualisation: spread comparison
spread_melted = spread_comparison.reset_index().melt(id_vars="index", var_name="Spread Type", value_name="Average")
spread_melted.rename(columns={"index": "Metric"}, inplace=True)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=spread_melted, x="Metric", y="Average", hue="Spread Type", palette=[primary, secondary])

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=10, fontweight="bold", padding=5)

plt.title("Average Engagement: High Spread vs Low Spread Posts", fontsize=14, fontweight="bold")
plt.xlabel("Engagement Metric", fontsize=11)
plt.ylabel("Average Count", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# visualisation: topic spread
plt.figure(figsize=(10, 6))
topic_spread = file.groupby("topic")["cross_platform_spread"].mean().sort_values(ascending=False)

ax = sns.barplot(x=topic_spread.head(10).index, y=topic_spread.head(10).values, palette=gradient, hue=topic_spread.head(10).index, legend=False)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=10, fontweight="bold", padding=5)

plt.title("Average Cross-Platform Spread by Topic", fontsize=14, fontweight="bold")
plt.xlabel("Topic", fontsize=11)
plt.ylabel("Average Spread Score", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Observations
- Posts that spread across more platforms have significantly higher engagement across all metrics.
- Topics like entertainment, gaming, and technology tend to have higher spread scores.
- However, some content performs very well on one platform without travelling much.

**What this suggests**: High engagement helps, but it does not guarantee cross-platform spread. Sentiment, topic, audience behaviour, and platform culture also matter.

### 6.1 Engagement and Virality Correlation

This step uses a correlation matrix to compare engagement metrics with viral coefficient and cross-platform spread.

In [ ]:
# virality correlation matrix
print("Correlation Between Engagement and Virality Indicators:")
virality_corr = file[["likes", "shares", "comments", "views", "viral_coefficient", "cross_platform_spread"]].corr().round(3)
virality_corr

In [ ]:
# visualisation: virality correlation heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(virality_corr, annot=True, cmap="PuBuGn", center=0, fmt=".2f", linewidths=1, cbar_kws={"shrink": 0.8})
plt.title("Correlation Between Engagement and Virality Indicators", fontsize=14, fontweight="bold")
plt.xlabel("Metric", fontsize=11)
plt.ylabel("Metric", fontsize=11)
plt.tight_layout()
plt.show()

### Observations
- Shares and viral coefficient show the strongest positive correlation, which makes sense – sharing is a direct driver of virality.
- Cross-platform spread also correlates moderately with all engagement metrics.
- Views and likes are less strongly associated with viral coefficient than shares are.

**What this suggests**: For content to become truly viral, encouraging shares [rather than just passive views or likes] is key.

### 7. Sentiment and Engagement Analysis

This section compares engagement by sentiment and checks whether the pattern changes across platforms.

**Questions:**
- Does positive content always perform better?
- How does sentiment affect engagement across different platforms?
- Is there a platform-specific sentiment advantage?

In [ ]:
print("Average Engagement by Sentiment:")
sentiment_engagement = file.groupby("sentiment_category")[["likes", "shares", "comments"]].mean().round(2)
print(sentiment_engagement)

In [ ]:
print("Average Likes by Sentiment and Platform:")
sentiment_platform = file.groupby(["platform", "sentiment_category"])["likes"].mean().unstack()
sentiment_platform.round(2)

In [ ]:
# visualisation: sentiment by platform
sentiment_melted = sentiment_platform.reset_index().melt(id_vars="platform", var_name="Sentiment", value_name="Average Likes")

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=sentiment_melted, x="platform", y="Average Likes", hue="Sentiment", palette=gradient[:len(sentiment_platform.columns)])

plt.title("Average Likes by Sentiment and Platform", fontsize=14, fontweight="bold")
plt.xlabel("Platform", fontsize=11)
plt.ylabel("Average Likes", fontsize=11)
plt.legend(title="Sentiment")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Observations
- Positive content generally receives more likes on most platforms.
- However, negative or neutral content can also perform well when it sparks discussion or strong reactions.
- The effect varies by platform: for example, Reddit sees less difference between sentiments than Instagram does.

**What this suggests**: Sentiment matters, but platform culture changes how sentiment works. A tone that gains attention on one platform may not perform the same way on another.

### 8. Account Authority Analysis

This section checks whether verified accounts and follower count are strongly related to engagement and viral coefficient.

**Questions:**
- Do verified and larger accounts always perform better?
- How strong is the relationship between follower count and engagement?
- Can smaller accounts still achieve virality?

In [ ]:
print("Average Engagement by Verification Status:")
verified_compare = file.groupby("verified")[["likes", "shares", "comments", "views"]].mean().round(2)
verified_compare.index = ["Not Verified", "Verified"]
print(verified_compare)

In [ ]:
# visualisation: verification status
verified_compare["Status"] = ["Not Verified", "Verified"]
verified_melted = verified_compare.reset_index(drop=True).melt(id_vars="Status", var_name="Metric", value_name="Average")
verified_melted.rename(columns={"verified": "Status"}, inplace=True)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=verified_melted, x="Status", y="Average", hue="Metric", palette=gradient[:4])

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=10, fontweight="bold", padding=5)

plt.title("Average Engagement by Verification Status", fontsize=14, fontweight="bold")
plt.xlabel("Verification Status", fontsize=11)
plt.ylabel("Average Count", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# visualisation: follower correlation
follower_corr = file[["followers", "likes", "shares", "comments", "views"]].corr().round(3)

plt.figure(figsize=(10, 6))
sns.heatmap(follower_corr, annot=True, cmap="Blues", fmt=".2f", linewidths=1)
plt.title("Correlation Between Follower Count and Engagement", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# visualisation: follower vs virality
plt.figure(figsize=(10, 6))
plt.scatter(file["followers"], file["viral_coefficient"], alpha=0.3, s=10, c=primary)
plt.title("Follower Count vs Viral Coefficient", fontsize=14, fontweight="bold")
plt.xlabel("Follower Count", fontsize=11)
plt.ylabel("Viral Coefficient", fontsize=11)
plt.xscale("log")
plt.tight_layout()
plt.show()

### Observations
- Verified accounts have higher average engagement across all metrics, but the difference is not as large as one might expect.
- Follower count correlates positively with engagement, but the relationship is moderate [r ≈ 0.4–0.6].
- Some smaller accounts still achieve high viral coefficients, as seen in the scatter plot.

**What this suggests**: A large audience gives an advantage, but it does not fully control virality. Content, timing, platform culture, and audience reaction still matter.

### 9. Virality Analysis

This section defines viral content as the top 10% of posts by viral coefficient. It then compares viral and non-viral content across engagement, topic, and sentiment.

**Questions:**
- What factors seem to separate viral posts from non-viral posts?
- Which topics and sentiments appear most among viral content?
- Is virality concentrated or distributed?

In [ ]:
# define viral content (top 10%)
viral_threshold = file["viral_coefficient"].quantile(0.9)
viral = file[file["viral_coefficient"] >= viral_threshold]
non_viral = file[file["viral_coefficient"] < viral_threshold]

print(f"Viral posts [top 10%]: {len(viral)}")
print(f"Non-viral posts: {len(non_viral)}")

In [ ]:
print("Viral vs Non-Viral Content Profile:")
viral_profile = pd.DataFrame({
    "Viral": viral[["likes", "shares", "comments", "views", "followers"]].mean(),
    "Non-Viral": non_viral[["likes", "shares", "comments", "views", "followers"]].mean()
}).round(2)
print(viral_profile)

In [ ]:
# visualisation: viral vs non-viral
viral_melted = viral_profile.reset_index().melt(id_vars="index", var_name="Type", value_name="Average")
viral_melted.rename(columns={"index": "Metric"}, inplace=True)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=viral_melted, x="Metric", y="Average", hue="Type", palette=[primary, secondary])

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", fontsize=10, fontweight="bold", padding=5)

plt.title("Viral vs Non-Viral Content Profile", fontsize=14, fontweight="bold")
plt.xlabel("Metric", fontsize=11)
plt.ylabel("Average Value", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# topic viral ratio
viral_topic_dist = viral["topic"].value_counts(normalize=True)
non_viral_topic_dist = non_viral["topic"].value_counts(normalize=True)

topic_viral_ratio = (viral_topic_dist / non_viral_topic_dist).sort_values(ascending=False)
print("Topic Viral Ratio [viral % / non-viral %]:")
print(topic_viral_ratio.head(10).round(2))

In [ ]:
print("Sentiment Distribution: Viral vs Non-Viral:")
viral_sentiment = viral["sentiment_category"].value_counts(normalize=True)
non_viral_sentiment = non_viral["sentiment_category"].value_counts(normalize=True)

sentiment_comparison = pd.DataFrame({
    "Viral": viral_sentiment,
    "Non-Viral": non_viral_sentiment
}).round(3)
print(sentiment_comparison)

In [ ]:
# visualisation: sentiment viral vs non-viral
sentiment_comp_melted = sentiment_comparison.reset_index().melt(id_vars="sentiment_category", var_name="Type", value_name="Proportion")

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=sentiment_comp_melted, x="sentiment_category", y="Proportion", hue="Type", palette=[primary, secondary])

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=10, fontweight="bold", padding=5)

plt.title("Sentiment Distribution: Viral vs Non-Viral Posts", fontsize=14, fontweight="bold")
plt.xlabel("Sentiment", fontsize=11)
plt.ylabel("Proportion", fontsize=11)
plt.tight_layout()
plt.show()

### Observations
- Viral posts have dramatically higher engagement [5–10 times] than non-viral posts.
- Entertainment, gaming, and technology topics are overrepresented among viral content.
- Positive sentiment is slightly more common in viral posts, but neutral and negative posts also go viral.

**What this suggests**: Virality often follows a winner-take-most pattern. A small share of posts collects a large share of attention, which is unfair but also very on-brand for social media.

### 10. Key Findings and Insights

What are the main patterns from the analysis?

#### 1. Platform attention profiles differ
- YouTube dominates view counts.
- TikTok generates high like volumes.
- Twitter [X] supports sharing behaviour.
- Reddit encourages discussion through comments.

#### 2. Topic and platform work together
- Entertainment, technology, and gaming appear strongly across platforms.
- Different platforms favour different topics.
- Discussion volume changes by topic and audience type.

#### 3. Timing has some influence
- Certain posting hours [late morning, early evening] are linked with higher engagement.
- Weekend posts may receive more engagement than weekday posts.
- Timing is useful, but it cannot save weak content.

#### 4. Virality is linked with spread and engagement
- Cross-platform spread is associated with stronger engagement.
- Shares are the strongest driver of viral coefficient.
- Follower count helps, but does not guarantee virality.

#### 5. The attention economy is not simple
- No single factor explains online success.
- Platform norms, topic type, timing, sentiment, and account authority all play a role.
- The interaction between these factors is likely more important than any one variable.

### 11. Statistical Summary

This final statistical summary brings together the main values for sample size, platform count, engagement, virality, and account authority.

In [ ]:
# statistical summary
sample_size = len(file)
platform_count = file["platform"].nunique()
topic_count = file["topic"].nunique()
mean_likes = file["likes"].mean()
mean_shares = file["shares"].mean()
mean_comments = file["comments"].mean()
mean_views = file["views"].mean()
mean_viral_coefficient = file["viral_coefficient"].mean()
mean_spread = file["cross_platform_spread"].mean()
verified_percentage = file[file["verified"] == True].shape[0] / sample_size * 100

print("KEY STATISTICS SUMMARY")
print("="*70)
print(f"\nSample Size: {sample_size} posts")
print(f"Platforms Covered: {platform_count}")
print(f"Topics Covered: {topic_count}")
print("\nAverage Engagement:")
print(f"  Likes: {mean_likes:.2f}")
print(f"  Shares: {mean_shares:.2f}")
print(f"  Comments: {mean_comments:.2f}")
print(f"  Views: {mean_views:.2f}")
print("\nVirality Indicators:")
print(f"  Average Viral Coefficient: {mean_viral_coefficient:.2f}")
print(f"  Average Cross-Platform Spread: {mean_spread:.2f}")
print("\nAccount Authority:")
print(f"  Verified Accounts: {verified_percentage:.1f}%")

### 12. Limitations

What should be kept in mind while reading this analysis?

1. **Dataset limitations**: the dataset may be synthetic or simplified, so it may not fully match real-world platform behaviour.

2. **Platform differences**: each platform measures engagement differently. A view on YouTube may not mean the same thing as a view on TikTok.

3. **Missing context**: the dataset may not include full user background or detailed post text for deeper interpretation.

4. **Correlation and causation**: the notebook observes relationships, but it does not prove cause and effect.

5. **Time period**: if the data covers a limited time frame, longer trends and seasonal changes may be missed.

6. **Language bias**: even when multiple languages are included, some languages may still be over-represented.

### 13. Conclusion

So what does the full analysis suggest?

### Summary

This analysis shows that online attention is shaped by a mix of platform design, topic type, timing, sentiment, and account authority. Different platforms reward different kinds of engagement, so a post that performs well in one place may not perform the same way elsewhere.

The strongest point from the notebook is that virality is not explained by one simple factor. High engagement, strong topics, and larger accounts may help, but they do not guarantee success. Cross-platform spread seems to be connected with stronger engagement, but correlation is not proof of causation.

### Implications

1. **For Content Creators**: Understanding platform-specific dynamics is more valuable than chasing universal virality. Content should be tailored to the platform where it will be shared.

2. **For Platform Analysts**: Engagement metrics tell different stories depending on the platform. A holistic view requires looking at likes, shares, comments, and views together rather than in isolation.

3. **For Researchers**: The attention economy is multi-dimensional. Future work should explore longitudinal patterns and control for confounding variables to better isolate causal mechanisms.

### Final Thoughts

The attention economy works like a very crowded school corridor. Everyone is talking, a few people are somehow louder than everyone else, and nobody is fully sure why one joke becomes famous. Data cannot explain everything, but it helps us see the patterns more clearly.
